**Actividad 2: Limpieza y análisis de datos con Pandas + NumPy**

**Contexto**

Un equipo de investigadores está recopilando información sobre observaciones de animales realizadas en diferentes países de Europa Central y Oriental.
Los datos fueron recolectados por diferentes personas y presentan problemas de calidad: valores faltantes, registros duplicados, formatos inconsistentes, errores en los datos y diferentes formas de representar algunas variables.
Antes de realizar cualquier análisis, es necesario diagnosticar y limpiar la información.
Tu trabajo como analista de datos será preparar esta base para que pueda ser utilizada posteriormente en un análisis de biodiversidad.

**1. Diagnóstico inicial**


In [1]:
from google.colab import files

uploaded = files.upload()


Saving animal_data_dirty1.csv to animal_data_dirty1.csv


In [2]:
import pandas as pd
import numpy as np
df = pd.read_csv("animal_data_dirty1.csv", sep=";")

print("Filas y columnas:", df.shape)
print("\nColumnas y tipo de dato: \n", df.dtypes)
print("\nValores faltantes por columna: \n", df.isnull().sum())
print("\nRegistros duplicados: \n",df.duplicated().sum())
unique_values = df["Country"].unique()
print("\nValores únicos en la columna Country: \n", unique_values)
value_counts = df["Country"].value_counts()
print("\nFrecuencia de valores en la columna Country: \n", value_counts)
unique_values = df["Animal type"].unique()
print("\nValores únicos en la columna Animal type: \n", unique_values)

Filas y columnas: (1011, 11)

Columnas y tipo de dato: 
 Animal type          object
Country              object
Weight kg           float64
Body Length cm      float64
Gender               object
Animal code         float64
Latitude            float64
Longitude           float64
Animal name          object
Observation date     object
Data compiled by     object
dtype: object

Valores faltantes por columna: 
 Animal type           20
Country               12
Weight kg             27
Body Length cm        27
Gender                19
Animal code         1011
Latitude              98
Longitude             98
Animal name          959
Observation date       0
Data compiled by       0
dtype: int64

Registros duplicados: 
 167

Valores únicos en la columna Country: 
 [nan 'Poland' 'Hungary' 'PL' 'Germany' 'Slovakia' 'Czech Republic' 'Czech'
 'Hungry' 'HU' 'Austria' 'Australia' 'CZ' 'DE' 'CC']

Frecuencia de valores en la columna Country: 
 Country
Poland            291
Germany           176
S

**Problemas de claidad**
1. Para la columna de Country encontramos varios paises repetidos pero escritos de diferente forma, ejemplo: "Poland" y "PL" es un mismo país. Por otro lado para Animal type tambien, ejemplo: red squirrel y red squirel es el mismo tipo de animal.
2. Tenemos 167 registro duplicados.
3. En algunas columnas faltan muchos datos, ejemplo, en animal code, faltan 1011 datos.

**2. Limpieza de datos**

In [3]:
clean = df.copy()
clean = clean.drop_duplicates()
print("Duplicados después de limpiar:", clean.duplicated().sum())
clean = clean.dropna(subset=["Animal type"])
animal_map = {"red squirel": "red squirrel", "red squirrell": "red squirrel", "lynx?": "lynx", "European bison™": "European bison", "European bisson": "European bison", "European buster": "European bison", "ledgehod": "hedgehog", "wedgehod": "hedgehog"}
clean["Animal type"] = clean["Animal type"].replace(animal_map)
country_map = {"PL": "Poland", "HU": "Hungary", "Hungry": "Hungary", "CZ": "Czech Republic", "Czech": "Czech Republic", "DE": "Germany", "Australia": "Austria", "CC": "Czech Republic"}
clean["Country"] = clean["Country"].replace(country_map)
clean["Country"] = clean["Country"].fillna("Unknown")
clean.loc[clean["Weight kg"] < 0, "Weight kg"] = np.nan
clean.loc[clean["Body Length cm"] < 0, "Body Length cm"] = np.nan
clean["Weight kg"] = clean.groupby("Animal type")["Weight kg"].transform(lambda x: x.fillna(x.median()))
clean["Body Length cm"] = clean.groupby("Animal type")["Body Length cm"].transform(lambda x: x.fillna(x.median()))
clean["Latitude"] = clean.groupby("Country")["Latitude"].transform(lambda x: x.fillna(x.median()))
clean["Longitude"] = clean.groupby("Country")["Longitude"].transform(lambda x: x.fillna(x.median()))
clean["Animal name"] = clean["Animal name"].fillna("Unknown")
clean["Gender"] = clean.groupby("Animal type")["Gender"].transform(lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else "not determined"))
clean["Observation date"] = pd.to_datetime(clean["Observation date"], format="mixed", dayfirst=True, errors="coerce")
clean = clean.drop(columns=["Animal code"])

print("Dimensiones finales:", clean.shape)
print("\nValores faltantes:",clean.isna().sum())
print("\nDuplicados:",clean.duplicated().sum())
print("\nTipo de dato corregido: \n", clean.dtypes)
print("\nTipo animal corregido:", clean["Animal type"].value_counts())

Duplicados después de limpiar: 0
Dimensiones finales: (825, 10)

Valores faltantes: Animal type         0
Country             0
Weight kg           0
Body Length cm      0
Gender              0
Latitude            0
Longitude           0
Animal name         0
Observation date    0
Data compiled by    0
dtype: int64

Duplicados: 0

Tipo de dato corregido: 
 Animal type                 object
Country                     object
Weight kg                  float64
Body Length cm             float64
Gender                      object
Latitude                   float64
Longitude                  float64
Animal name                 object
Observation date    datetime64[ns]
Data compiled by            object
dtype: object

Tipo animal corregido: Animal type
red squirrel      413
hedgehog          278
lynx               71
European bison     63
Name: count, dtype: int64


**3. NumPy**

In [5]:
clean["Size category"] = np.where(clean["Weight kg"] >= 100, "large", "small")
print(clean["Size category"].value_counts())
media_peso = np.nanmean(clean["Weight kg"])
print("Peso promedio:", media_peso)
mediana_peso = np.nanmedian(clean["Weight kg"])
print("Mediana del peso:", mediana_peso)
desviacion_peso = np.nanstd(clean["Weight kg"])
print("Desviación estándar del peso:", desviacion_peso)

Size category
small    765
large     60
Name: count, dtype: int64
Peso promedio: 47.352241212121214
Mediana del peso: 0.36
Desviación estándar del peso: 169.54809445207127


La media representa el peso promedio de los animales observados y la desviación estándar indica cuánto varían los pesos alrededor de la media, en este la diferencia entre media y mediana es grande porque hayuna especie muy pesada.

**4. Análisis con groupby()**

In [16]:
observaciones_animal = clean.groupby("Animal type").size()
print(observaciones_animal)
peso_por_animal = clean.groupby("Animal type")["Weight kg"].mean()
print("\nPeso", peso_por_animal)
peso_por_pais = clean.groupby("Country")["Weight kg"].mean()
print("\nPeso", peso_por_pais.round(2))
genero_animal = clean.groupby(["Animal type", "Gender"]).size()
print("\n", genero_animal)

Animal type
European bison     63
hedgehog          278
lynx               71
red squirrel      413
dtype: int64

Peso Animal type
European bison    588.396825
hedgehog            0.797482
lynx               23.267606
red squirrel        0.297576
Name: Weight kg, dtype: float64

Peso Country
Austria             2.05
Czech Republic     74.50
Germany             0.46
Hungary             2.88
Poland            130.00
Slovakia           32.45
Name: Weight kg, dtype: float64

 Animal type     Gender        
European bison  female             35
                male               26
                not determined      2
hedgehog        female            136
                male              140
                not determined      2
lynx            female             37
                male               34
red squirrel    female            191
                male              221
                not determined      1
dtype: int64


**5. Interpretación de resultados**

*   El red squirrel es el animal con mayor cantidad de observaciones contando con 413 registros, por otro lado, el European bison cuenta con 63. Esto indica que el dataset presenta una distribución desigual de observaciones entre las especies.
*   El European bison presenta el mayor peso promedio con 588.40 kgaproximadamente, mientras que el red squirrel presenta un promedio cercano a 0.30 kg. Esta diferencia refleja las grandes variaciones de tamaño corporal entre las especies incluidas en el dataset.
*   El dataset original contenía 167 registros duplicados, valores faltantes, categorías inconsistentes y valores numéricos inválidos. Después de aplicar la limpieza, se eliminaron los duplicados y se estandarizaron las categorías, obteniendo una base de datos de 825 observaciones preparada para análisis posteriores.
*   El peso promedio varía entre países pero el país con mayor presencia de european bison puede presentar un peso promedio mucho mayor que otro donde predomine otro animal.



